In [2]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
print("Added to path:", os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import numpy as np
import torch, random
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, TrainingArguments, Trainer, default_data_collator

from src.dataset.data_loader import load_competition_data
from src.models.LoRA_MC import (
    OPTIONS, LABEL_TO_ID, ID_TO_LABEL, DEFAULT_MODEL_NAME,
    build_hf_mc_dataset, build_lora_mc_model, mc_compute_metrics, MC_LoRA_Predictor,
)
from src.inference.LoRA_MC_inference import run_lora_mc_predictions, save_submission

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Added to path: /teamspace/studios/this_studio/genAI-project-24f1002692


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).


Using device: cuda


In [3]:
# Load competition data, encode answer letters (A-E) as numeric labels
train_dataset, test_dataset, sample_submission = load_competition_data()

train_df = train_dataset.copy()
train_df['label'] = train_df['answer'].map(LABEL_TO_ID)
test_df = test_dataset.copy()

print("Label mapping:", LABEL_TO_ID)
print(train_df[['answer', 'label']].head())

Label mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
  answer  label
0      B      1
1      A      0
2      C      2
3      B      1
4      A      0


In [4]:
# Prompt-level train/val split (prevents the same question leaking across both sets)
prompt_answer = train_df.groupby('prompt')['answer'].first()
unique_prompts = prompt_answer.index.values

train_prompts, val_prompts = train_test_split(
    unique_prompts, test_size=0.1, random_state=SEED, stratify=prompt_answer.values
)

train_split = train_df[train_df['prompt'].isin(train_prompts)].reset_index(drop=True)
val_split   = train_df[train_df['prompt'].isin(val_prompts)].reset_index(drop=True)

overlap = set(train_split['prompt']) & set(val_split['prompt'])
print(f"Train rows: {len(train_split)}  |  Val rows: {len(val_split)}  |  Prompt overlap: {len(overlap)}")

Train rows: 1801  |  Val rows: 199  |  Prompt overlap: 0


In [5]:
# Tokenizer + LoRA-wrapped bert-base-uncased (r=8, alpha=16, target=[query,value])
MC_MAX_LENGTH = 128

MC_tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL_NAME)
MC_model = build_lora_mc_model(DEFAULT_MODEL_NAME)
MC_model.to(device)

hf_mc_train = build_hf_mc_dataset(train_split, MC_tokenizer, max_length=MC_MAX_LENGTH)
hf_mc_val   = build_hf_mc_dataset(val_split, MC_tokenizer, max_length=MC_MAX_LENGTH)

print(hf_mc_train)
print(hf_mc_val)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1801
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 199
})


In [6]:
# Fine-tune with HF Trainer (tracks accuracy + MAP@3, keeps best-MAP@3 checkpoint)
MC_training_args = TrainingArguments(
    output_dir="./m4_lora_MC_model",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=18,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

MC_trainer = Trainer(
    model=MC_model,
    args=MC_training_args,
    train_dataset=hf_mc_train,
    eval_dataset=hf_mc_val,
    data_collator=default_data_collator,
    compute_metrics=mc_compute_metrics,
)

MC_train_result = MC_trainer.train()
print(MC_train_result)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Map3
1,2.877159,1.327911,0.437186,0.632328
2,2.143822,0.913794,0.653266,0.775544
3,1.621977,0.631622,0.758794,0.849246
4,1.135044,0.434742,0.849246,0.906198
5,0.956325,0.306770,0.899497,0.929648
6,0.831974,0.251981,0.914573,0.947236
7,0.697927,0.178774,0.924623,0.956449
8,0.684689,0.151246,0.959799,0.976549
9,0.547131,0.113778,0.969849,0.984087
10,0.446536,0.110962,0.974874,0.986600


TrainOutput(global_step=2034, training_loss=0.8581879896165816, metrics={'train_runtime': 704.0782, 'train_samples_per_second': 46.043, 'train_steps_per_second': 2.889, 'total_flos': 1.069862989653504e+16, 'train_loss': 0.8581879896165816, 'epoch': 18.0})


In [7]:
# Evaluate + save the best LoRA adapter checkpoint
MC_eval_metrics = MC_trainer.evaluate()
print("Final validation metrics:", MC_eval_metrics)

MC_trainer.save_model("./m4_lora_mc_model_best")
print("Best LoRA multiple-choice model saved to ./m4_lora_mc_model_best")

Training Loss,Validation Loss,Epoch,Accuracy,Map3
0.261835,0.060093,18,0.974874,0.987437


Final validation metrics: {'eval_loss': 0.06009255722165108, 'eval_accuracy': 0.9748743718592965, 'eval_map3': 0.9874371859296482}
Best LoRA multiple-choice model saved to ./m4_lora_mc_model_best


In [8]:
# Load the saved LoRA adapter (merged into the base model), predict on test.csv, save submission
predictor = MC_LoRA_Predictor(
    lora_checkpoint_dir="./m4_lora_mc_model_best",
    base_model_name=DEFAULT_MODEL_NAME,
    max_length=MC_MAX_LENGTH,
    device=device,
)

submission = run_lora_mc_predictions(predictor, test_df, id_col="id", batch_size=32)
save_submission(submission, out_path="submission_lora_mc.csv")
submission.head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Predicting on test set: 100%|█

Saved: submission_lora_mc.csv (500 rows)


,ID,Prediction
0,1,A B D
1,2,B A C
2,3,B E D
3,4,E C D
4,5,C A D
5,6,D B A
6,7,E C A
7,8,B A C
8,9,C D E
9,10,B D A


: 